# Forecasting — Projeção de Empregos no Telemarketing

Este notebook utiliza o Prophet para projetar a tendência de empregos 
no setor de telemarketing até 2030.

**Input:** output/caged_telemarketing_tratado.csv  
**Output:** output/forecast_telemarketing.csv

**Modelo:** Prophet (Meta/Facebook)  
**Horizonte:** 2025 até 2030

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

# Carregar dado tratado
df = pd.read_csv('../output/caged_telemarketing_tratado.csv')
df['data'] = pd.to_datetime(df['data'])

print(f"Dados carregados: {df.shape}")
print(f"Período: {df['data'].min().strftime('%b/%Y')} até {df['data'].max().strftime('%b/%Y')}")

Importing plotly failed. Interactive plots will not work.


Dados carregados: (187, 5)
Período: Jan/2010 até Jul/2025


In [2]:
# Preparar dados no formato que o Prophet exige
df_prophet = df[['data', 'estoque_acumulado']].rename(columns={
    'data': 'ds',
    'estoque_acumulado': 'y'
})

print("Dados no formato Prophet:")
print(df_prophet.head())
print(f"\nShape: {df_prophet.shape}")

Dados no formato Prophet:
          ds      y
0 2010-01-01   1998
1 2010-02-01   5403
2 2010-03-01  13842
3 2010-04-01  18153
4 2010-05-01  24366

Shape: (187, 2)


In [3]:
# Configurar e treinar o modelo
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)

modelo.fit(df_prophet)

print("Modelo treinado com sucesso!")

13:41:09 - cmdstanpy - INFO - Chain [1] start processing
13:41:09 - cmdstanpy - INFO - Chain [1] done processing


Modelo treinado com sucesso!


In [4]:
# Criar datas futuras até dezembro de 2030
futuro = modelo.make_future_dataframe(
    periods=65,
    freq='MS'
)

# Gerar previsão
forecast = modelo.predict(futuro)

print(f"Previsão gerada!")
print(f"Período previsto: {forecast['ds'].iloc[-1].strftime('%b/%Y')}")
print(f"\nColunas principais:")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10))

Previsão gerada!
Período previsto: Dec/2030

Colunas principais:
            ds           yhat     yhat_lower     yhat_upper
242 2030-03-01  268235.104910  233430.834646  299834.293522
243 2030-04-01  267882.031474  234208.928416  303658.360984
244 2030-05-01  266452.867839  231623.728482  298999.774506
245 2030-06-01  265028.045124  229305.725478  298286.902205
246 2030-07-01  263050.955817  226597.196374  297773.539670
247 2030-08-01  262177.588448  225979.236887  298238.871266
248 2030-09-01  262796.026570  224424.914209  298793.736770
249 2030-10-01  264343.141551  226792.942805  298261.166312
250 2030-11-01  269563.880479  229276.912547  305666.038148
251 2030-12-01  268945.984270  229299.299650  306617.895048


In [5]:
# Filtrar apenas dados a partir do pico (nov/2021)
df_recente = df_prophet[df_prophet['ds'] >= '2021-11-01'].copy()

print(f"Dados para cenário recente: {len(df_recente)} registros")
print(f"Período: {df_recente['ds'].min().strftime('%b/%Y')} até {df_recente['ds'].max().strftime('%b/%Y')}")

# Treinar modelo com dados recentes
modelo_recente = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.1
)

modelo_recente.fit(df_recente)

# Gerar previsão
futuro_recente = modelo_recente.make_future_dataframe(
    periods=65,
    freq='MS'
)

forecast_recente = modelo_recente.predict(futuro_recente)

print(f"\nModelo recente treinado!")
print(f"\nPrevisão para Dez/2030:")
ultimo = forecast_recente[forecast_recente['ds'] == '2030-12-01'].iloc[0]
print(f"  Central: {ultimo['yhat']:,.0f}")
print(f"  Mínimo:  {ultimo['yhat_lower']:,.0f}")
print(f"  Máximo:  {ultimo['yhat_upper']:,.0f}")

13:42:58 - cmdstanpy - INFO - Chain [1] start processing


Dados para cenário recente: 45 registros
Período: Nov/2021 até Jul/2025


13:42:58 - cmdstanpy - INFO - Chain [1] done processing



Modelo recente treinado!

Previsão para Dez/2030:
  Central: 342,476
  Mínimo:  -524,230
  Máximo:  1,173,258
